# ML-02 - Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhnehh/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames the project before modeling. The numbers below come from the bundled anonymized starter dataset. The wording is intentionally careful: this is observational decision support, not causal proof and not a reconstruction of Google's algorithm.

**Repo references used:** `docs/data-dictionary.md`, `docs/ml-intern-dataset-and-lane-guide.md`, and `skills/framing-ml-problems/SKILL.md`.

## 1. My lane (or freestyle) and why

*Name your lane - or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane:** CTR / Engagement Opportunity Scoring

**Why:** I will identify pages that already receive meaningful search visibility but capture fewer clicks than comparable pages at similar search positions. The main output will be a position- and volume-adjusted review queue for an editorial or content-growth team. This is more useful than applying one universal CTR threshold, because CTR changes strongly with search position and becomes unstable when impression counts are small. Engagement metrics will be used only as secondary context after a click, not as proof that a CTR recommendation is correct.

In [1]:
# The lane is supported by observable fields in the starter dataset.
required_columns = {
    'content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'ctr',
    'avg_position', 'position_tier', 'main_intent', 'content_type',
    'engagement_rate', 'scroll_rate'
}

# df is loaded in Section 3 below; this cell intentionally records the planned fields.
print('Planned lane fields:', ', '.join(sorted(required_columns)))

Planned lane fields: avg_position, clicks_90d, client_id, content_id, content_type, ctr, engagement_rate, impressions_90d, main_intent, position_tier, scroll_rate


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

- **Research question:** Which visible pages have lower observed CTR than comparable pages at similar search positions and should therefore be reviewed first?
- **Decision improved:** Which existing pages should enter the editorial review queue first?
- **Who acts:** An SEO analyst, editor, or content-growth team.
- **Action:** Manually inspect title/meta description, snippet structure, search-intent match, and the opening section; then monitor or test any change.
- **Unit of analysis:** One pseudonymized page (`content_id`).
- **False-positive cost:** Editorial time is spent on a page whose CTR gap is mostly noise or whose snippet cannot be improved through the available action.
- **False-negative cost:** A high-visibility page with a credible click-capture gap is left unreviewed.
- **Important boundary:** The system recommends review. It does not automatically rewrite or publish content.

In [2]:
# A compact written frame that can be reused later in the report.
project_frame = {
    'decision': 'which pages should be reviewed first',
    'actor': 'SEO/editorial/content-growth team',
    'output': 'ranked CTR opportunity queue with reason codes',
    'false_positive_cost': 'wasted editorial review time',
    'false_negative_cost': 'missed high-visibility review opportunity',
}
project_frame

{'decision': 'which pages should be reviewed first',
 'actor': 'SEO/editorial/content-growth team',
 'output': 'ranked CTR opportunity queue with reason codes',
 'false_positive_cost': 'wasted editorial review time',
 'false_negative_cost': 'missed high-visibility review opportunity'}

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
from pathlib import Path
import pandas as pd

# Works in Colab, a cloned repo, and the downloadable notebook package.
DATA_CANDIDATES = [
    Path('/content/content_refresh_anonymized.csv'),
    Path('/mnt/data/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        df = pd.read_csv(candidate)
        break
else:
    DATA_URL = (
        'https://raw.githubusercontent.com/muhnehh/'
        'flyrank-ml-internship-starter/main/data/raw/'
        'content_refresh_anonymized.csv'
    )
    df = pd.read_csv(DATA_URL)

print(f'Dataset loaded: {df.shape[0]:,} pages x {df.shape[1]} columns')
print(f'Clients: {df["client_id"].nunique():,}')
print(f'Unique content IDs: {df["content_id"].nunique():,}')


Dataset loaded: 30,000 pages x 44 columns
Clients: 32
Unique content IDs: 30,000


In [4]:
import numpy as np

# Use a clear visibility floor for this first descriptive check.
# avg_position == 0 means "no position data" in this dataset, so those rows are excluded.
eligible = df[
    (df['impressions_90d'] >= 1_000)
    & (df['avg_position'] > 0)
].copy()

# A transparent discovery baseline: median CTR among eligible pages in the same position tier.
# CTR is stored in percentage points: 0.76 means 0.76%, not 76%.
tier_expected_ctr = eligible.groupby('position_tier')['ctr'].median()
eligible['expected_ctr_tier'] = eligible['position_tier'].map(tier_expected_ctr)
eligible['ctr_gap_pp'] = eligible['expected_ctr_tier'] - eligible['ctr']
eligible['estimated_click_gap_90d'] = (
    eligible['ctr_gap_pp'].clip(lower=0) / 100
    * eligible['impressions_90d']
)

candidates = eligible[eligible['ctr_gap_pp'] > 0].copy()

print(f'Dataset scope: {len(df):,} pages across {df.client_id.nunique():,} clients.')
print(f'Eligible visible pages: {len(eligible):,} have >=1,000 impressions and valid position data.')
print(f'Descriptive candidates: {len(candidates):,} are below the median CTR of their position tier.')
print(
    'Directional aggregate gap under this simple baseline: '
    f'{candidates.estimated_click_gap_90d.sum():,.0f} clicks over 90 days.'
)
print('\nMedian CTR by position tier among eligible pages (percentage points):')
print(tier_expected_ctr.sort_values(ascending=False).to_string())

Dataset scope: 30,000 pages across 32 clients.
Eligible visible pages: 13,512 have >=1,000 impressions and valid position data.
Descriptive candidates: 6,562 are below the median CTR of their position tier.
Directional aggregate gap under this simple baseline: 63,488 clicks over 90 days.

Median CTR by position tier among eligible pages (percentage points):
position_tier
page_1      0.24
top_3       0.23
striking    0.19
page_3_5    0.10
deep        0.00


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) - and what it never will (causal proof, 'predicting Google').*

### What I can claim

- **Observed scope:** The starter dataset contains 30,000 pseudonymized pages from 32 clients, with one row per page.
- **Observed opportunity pattern:** Among pages with at least 1,000 impressions and valid position data, thousands sit below the median CTR of other eligible pages in the same position tier.
- **Directional decision support:** A position- and volume-adjusted score can organize pages for manual review.
- **Transparent comparison:** A simple comparable-page baseline can be built first and evaluated before adding ML.

### What I cannot claim

- **No causal guarantee:** The observational data cannot prove that changing a title, meta description, or introduction will cause a particular traffic increase.
- **No Google reverse-engineering:** The analysis does not reconstruct or predict Google's ranking algorithm.
- **No direct competitor conclusion:** The anonymized data does not show the exact SERP layout, competing results, ads, snippets, or query wording.
- **No automatic content-quality judgment:** A low CTR gap is a review signal, not proof that a page is poor.
- **No guaranteed click total:** The estimated click gap is a directional prioritization quantity based on a simple baseline, not promised future traffic.

In [5]:
# Final public-safety check for this notebook.
assert df['content_id'].nunique() == len(df)
assert (df['avg_position'] == 0).sum() == 1205
assert np.isclose(
    df.loc[df['impressions_90d'] > 0, 'ctr'],
    100 * df.loc[df['impressions_90d'] > 0, 'clicks_90d']
    / df.loc[df['impressions_90d'] > 0, 'impressions_90d'],
    atol=0.011,
).all()
print('Checks passed: one row per content_id, position-zero sentinel recognized, CTR scale verified.')

Checks passed: one row per content_id, position-zero sentinel recognized, CTR scale verified.


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled - markdown thinking and code that backs it.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, raw keywords, or private queries appear anywhere.
- [x] Claims use careful words: observed, measured, directional, decision-support.
- [ ] Save this file to `work/notebooks/w01_research_question.ipynb` in your repo and submit the repo URL on the assignment card.